In [ ]:
# Cài đặt các thư viện cần thiết cho Backend và AI Marker
!pip install fastapi uvicorn pyngrok marker-pdf python-multipart nest-asyncio

In [ ]:
"""
LƯU Ý: FILE NÀY CHỈ ĐƯỢC CHẠY TRÊN COLAB TRÁNH TRÀN RAM
""" 
import nest_asyncio
from pyngrok import ngrok
import uvicorn
from fastapi import FastAPI, UploadFile, File, BackgroundTasks
import os
import shutil
import glob
import subprocess

NGROK_TOKEN = "3CTWL2LB2xz9jAo4Z8xbUR0BZOE_5XD4KbjPQGpSHkHv84tB6"
ngrok.set_auth_token(NGROK_TOKEN)

app = FastAPI()

def run_marker(input_path, output_dir):
    result = subprocess.run(
        ["marker_single", input_path, "--output_dir", output_dir],
        stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True
    )
    # Nếu sập, trả lỗi về file txt
    if result.returncode != 0:
        with open(os.path.join(output_dir, "ERROR.txt"), "w") as f:
            f.write(result.stderr)

@app.post("/convert")
async def convert_pdf(background_tasks: BackgroundTasks, file: UploadFile = File(...)):
    # Tạo thư mục riêng cho từng file
    file_id = file.filename.replace(".pdf", "")
    input_path = f"input_{file_id}.pdf"
    output_dir = f"output_{file_id}"

    if os.path.exists(output_dir):
        shutil.rmtree(output_dir)
    os.makedirs(output_dir)

    with open(input_path, "wb") as f:
        f.write(await file.read())

    # 2. Quăng việc cho công nhân chạy ngầm (Background)
    background_tasks.add_task(run_marker, input_path, output_dir)

    # 3. Trả lời Python NGAY LẬP TỨC để Ngrok không bị Timeout
    return {"status": "processing", "file_id": file_id}

@app.get("/result/{file_id}")
def get_result(file_id: str):
    """API để Python gọi sang hỏi thăm tình hình"""
    output_dir = f"output_{file_id}"

    # Tình huống 1: Phát hiện tâm thư (Lỗi)
    error_file = os.path.join(output_dir, "ERROR.txt")
    if os.path.exists(error_file):
        with open(error_file, "r") as f:
            return {"status": "error", "message": f.read()}

    # Tình huống 2: Thấy file Markdown ra lò (Thành công)
    md_files = glob.glob(f"{output_dir}/**/*.md", recursive=True)
    if md_files:
        with open(md_files[0], "r", encoding="utf-8") as f:
            return {"status": "done", "text": f.read()}

    # Tình huống 3: Không có lỗi, chưa có file -> Đang làm
    return {"status": "processing"}

ngrok.kill()
public_url = ngrok.connect(8000).public_url
print(f"\n🚀 LINK API MỚI: {public_url}\n")

# Chạy server
nest_asyncio.apply()
config = uvicorn.Config(app, host="0.0.0.0", port=8000, loop="asyncio")
server = uvicorn.Server(config)
await server.serve()